In [10]:
import numpy as np, pandas as pd, os
import sys

sys.path.insert(0, os.path.abspath("."))

import features as F

MODE = "url"                 # "host" = main model, "url" = comparison model
OUT = f"../data/features/{MODE}"
os.makedirs(OUT, exist_ok=True)

train = pd.read_csv("../data/splits/train.csv")
val   = pd.read_csv("../data/splits/val.csv")

if MODE == "host":
    n_tr, n_va = len(train), len(val)
    train, conf_tr = F.to_host_dataset(train)
    val,   conf_va = F.to_host_dataset(val)
    print(f"Train: {n_tr} URLs -> {len(train)} unique hosts "
          f"({conf_tr['host'].nunique()} conflicting hosts dropped)")
    print(f"Val:   {n_va} URLs -> {len(val)} unique hosts "
          f"({conf_va['host'].nunique()} conflicting hosts dropped)")

print("Train labels:", train["label"].value_counts().to_dict())
print("Val labels:  ", val["label"].value_counts().to_dict())

Train labels: {0: 94394, 1: 70364}
Val labels:   {0: 20228, 1: 15078}


In [11]:
lens = train["URL"].map(lambda u: len(F.to_text(u, MODE)))
print(lens.describe(percentiles=[.5, .9, .95, .99]).round(1))
for L in ([32, 48, 64, 80] if MODE == "host" else [100, 150, 200]):
    print(f"max_len={L}: covers {(lens <= L).mean():.1%}")

count    164758.0
mean         27.6
std          41.3
min           7.0
50%          21.0
90%          43.0
95%          68.0
99%         141.0
max        6089.0
Name: URL, dtype: float64
max_len=100: covers 98.3%
max_len=150: covers 99.0%
max_len=200: covers 99.5%


In [12]:
MAX_LEN = 100 # set from Cell 1

vocab = F.build_vocab(train["URL"], mode=MODE)
print("Vocab size (incl. PAD and UNK):", len(vocab) + 2)

X_val_tmp = F.encode(val["URL"], vocab, MAX_LEN, MODE)
print(f"UNK rate on val: {(X_val_tmp == F.UNK).sum() / (X_val_tmp != F.PAD).sum():.4%}")

F.save_vocab(vocab, MAX_LEN, MODE, f"{OUT}/vocab.json")

Vocab size (incl. PAD and UNK): 67
UNK rate on val: 0.0000%


In [13]:
X_train = F.encode(train["URL"], vocab, MAX_LEN, MODE)
X_val   = F.encode(val["URL"],   vocab, MAX_LEN, MODE)

np.save(f"{OUT}/X_train_chars.npy", X_train)
np.save(f"{OUT}/X_val_chars.npy",   X_val)
np.save(f"{OUT}/y_train.npy", train["label"].values)
np.save(f"{OUT}/y_val.npy",   val["label"].values)

print(X_train.shape, X_val.shape)
print("Original:", train["URL"].iloc[0])
print("Decoded: ", F.decode(X_train[0], vocab))

(164758, 100) (35306, 100)
Original: https://www.southbankmosaics.com
Decoded:  www.southbankmosaics.com


In [14]:
lex_train = F.lexical_frame(train["URL"], MODE)
lex_val   = F.lexical_frame(val["URL"],   MODE)
assert list(lex_train.columns) == list(lex_val.columns)

lex_train.to_csv(f"{OUT}/lex_train.csv", index=False)
lex_val.to_csv(f"{OUT}/lex_val.csv",     index=False)

lex_train.assign(label=train["label"].values).groupby("label").mean().T.round(3)

label,0,1
len,19.231,38.909
host_len,15.231,22.608
sld_len,10.837,11.753
n_dots,2.162,2.356
n_hyphens,0.082,0.712
n_digits,0.049,4.431
is_ip,0.000,0.006
n_subdomains,0.029,0.414
entropy,3.376,3.867
n_suspicious,0.004,0.100
